In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from spacy.lang.en.stop_words import STOP_WORDS


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# root_path = "/content/drive/My Drive/notebooks"
# # For Google colab only

df = pd.read_csv(f"../new_clean_sm_100000.csv")
df = df[df['reviewText'].notna()]
df = df[~df['reviewText'].str.contains(".jpg|.png|.jpeg|.tiff|.gif|.bmp|.heif", regex=True, na=False)]

Split data

In [ ]:
### Convert to a two class problem -  optional

df = df[df['overall'] != 2]
df = df[df['overall'] != 4]
df.loc[df['overall'] == 1, 'overall'] = 0
df.loc[df['overall'] == 3, 'overall'] = 1
df.loc[df['overall'] == 5, 'overall'] = 2

In [ ]:
X = df.reviewText.values
y = df.overall.values

In [ ]:
num = len(df)
X, y  = X[:num], y[:num]
# le = LabelEncoder()
# y = le.fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
def train_predict(pipline):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    score = classification_report(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    accuracy = accuracy_score(y_test, y_pred)
    print(score)
    print(f"f1 score is {f1}, accuracy is {accuracy}")

def train_predict_all(pipeline):
    out = pipeline.fit(X, y).cv_results_
    results_df = pd.DataFrame({'rank': out['rank_test_score'],
                          'params': out['params'],
                           'cv score (mean)': out['mean_test_score']})
    results_df = results_df.sort_values(by=['rank'], ascending=True)
    pd.set_option('display.max_colwidth',100)
    display(HTML(results_df.to_html()))
    return out
    # score = classification_report(y_test, y_pred)
    # f1 = f1_score(y_test, y_pred, average='macro')
    # accuracy = accuracy_score(y_test, y_pred)
    # print(score)
    # print(f"f1 score is {f1}, accuracy is {accuracy}")


Vectorizers


In [ ]:


count_vectoriser = Pipeline([
                ('countVectoriser', CountVectorizer())
            ])

tfidf_vectoriser = Pipeline([
                ('tfidfVectoriser', TfidfVectorizer(stop_words=STOP_WORDS, ngram_range = (1,2)
                                                    ))
            ])


### Stantard Models

Naive Bayes -  Fine Tune


In [ ]:
param_grid = {'vectoriser__tfidfVectoriser__ngram_range': [(1,1),(1,2)],
              'classifier__classifier__alpha': [1e-5, 1e-4, 1e-2, 1e-1,1]}


# param_grid = {'vectoriser__tfidfVectoriser__ngram_range': [(1,1)],
#               'classifier__classifier__alpha': [1e-5]}

naive_bayes = Pipeline([
    ('classifier', MultinomialNB()),
])

pipeline = Pipeline([
    ('vectoriser', tfidf_vectoriser),
    ('classifier', naive_bayes)
])


In [ ]:
gs_mnb = GridSearchCV(pipeline, param_grid, cv=5, verbose=2, n_jobs=3)
final_results = train_predict_all(gs_mnb)

Max Entropy

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:

max_ent = Pipeline([
    ('classifier', LogisticRegression(penalty='l2', C=1.0)),
])

pipeline = Pipeline([
    ('vectoriser', tfidf_vectoriser),
    ('classifier', max_ent)
])

In [ ]:
train_predict(pipeline)

Logistic Regression - Fine Tune

In [ ]:
param_grid={"classifier__classifier__C":np.logspace(-3,3,7), "classifier__classifier__penalty":["l1","l2"],
             'vectoriser__tfidfVectoriser__ngram_range': [(1,1),(1,2)]}

logistic_regression = Pipeline([
    ('classifier', LogisticRegression(n_jobs=-1)),
])

pipeline = Pipeline([
    ('vectoriser', tfidf_vectoriser),
    ('classifier', logistic_regression)
])

In [ ]:
gs_mnb = GridSearchCV(pipeline, param_grid, cv=5, verbose=2)
results = train_predict_all(gs_mnb)




 Decision Tree - Fine Tune

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
param_grid = {"decisiontreeregressor__min_samples_leaf" : [1, 5, 10, 20, 50, 100],
               'vectoriser__tfidfVectoriser__ngram_range': [(1,1),(1,2)]}


decision_tree = Pipeline([
    ('classifier', DecisionTreeClassifier(max_depth=25, n_jobs=-1)),
])

pipeline = Pipeline([
    ('vectoriser', tfidf_vectoriser),
    ('classifier', decision_tree)
])

In [ ]:
gs_mnb = GridSearchCV(pipeline, param_grid, cv=5, verbose=2)
results = train_predict_all(gs_mnb)

### Bagging Models

Random Forest - Fine Tune

In [ ]:
# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
# Number of features to consider at every split
max_features = ['auto', 'sqrt']
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
# Method of selecting samples for training each tree
bootstrap = [True, False]# Create the random grid
random_grid = {'classifier__classifier__n_estimators': n_estimators,
               'classifier__classifier__max_features': max_features,
               'classifier__classifier__min_samples_split': min_samples_split,
               'classifier__classifier__min_samples_leaf': min_samples_leaf,
               'classifier__classifier__bootstrap': bootstrap}

random_forest = Pipeline([
    ('classifier', RandomForestClassifier(max_depth=20, n_jobs=-1)),
])

pipeline = Pipeline([
    ('vectoriser', tfidf_vectoriser),
    ('classifier', random_forest)
])

In [ ]:
train_predict(pipeline)

### Boosting Models



XGBoost - Fine Tune


In [ ]:
# !wget https://s3-us-west-2.amazonaws.com/xgboost-wheels/xgboost-0.81-py2.py3-none-manylinux1_x86_64.whl
# !pip uninstall xgboost --yes
# !pip install xgboost-0.81-py2.py3-none-manylinux1_x86_64.whl

# For Google Colab Only

In [ ]:
from xgboost import XGBClassifier

In [ ]:

xgb_clf = XGBClassifier()# = "gpu_hist")
# Create parameter grid
parameters = {"classifier__classifier__learning_rate": [0.1, 0.01, 0.001],
               "classifier__classifier__gamma" : [0.01, 0.1, 0.3, 0.5, 1, 1.5, 2],
               "classifier__classifier__max_depth": [2, 4, 7, 10],
               "classifier__classifier__colsample_bytree": [0.3, 0.6, 0.8, 1.0],
               "classifier__classifier__reg_alpha": [0, 0.5, 1],
               "classifier__classifier__reg_lambda": [1, 1.5, 2, 3, 4.5],
               "classifier__classifier__min_child_weight": [1, 3, 5, 7],
               "classifier__classifier__n_estimators": [100, 250, 500, 1000]}


pipeline = Pipeline([
    ('vectoriser', tfidf_vectoriser),
    ('classifier', xgb_clf)
])

In [ ]:
# gs_mnb = RandomizedSearchCV(pipeline, parameters, scoring = "f1_micro",
#                              cv = 5, verbose = 2)
# results = train_predict_all(gs_mnb)
train_predict(xgb_clf)

AdaBoost


In [ ]:
from sklearn.ensemble import AdaBoostClassifier


In [ ]:

param_grid = {
 'classifier__classifier__n_estimators': [50, 100],
 'classifier__classifier__learning_rate' : [0.01,0.05,0.1,0.3,1],
 'classifier__classifier__algorithm' : ['SAMME', 'SAMME.R']
 }


adaBoost = Pipeline([
    ('classifier', AdaBoostClassifier()),
])

pipeline = Pipeline([
    ('vectoriser', tfidf_vectoriser),
    ('classifier', adaBoost)
])


In [ ]:
gs_mnb = RandomizedSearchCV(pipeline, param_grid, cv=5, verbose=2, n_jobs=-1)
results = train_predict_all(gs_mnb)


In [ ]:
train_predict(pipeline)



In [ ]:
from pprint import pprint
pprint(gs_mnb.estimator.get_params().keys())